# Export DistilBERT SQuAD to TensorRT

This notebook exports `distilbert-base-cased-distilled-squad` to ONNX and builds `distilbert_squad_trt/1/model.plan` with `trtexec`.

Build this TensorRT plan in the same Triton image and on the same GPU class that will serve it.

In [ ]:
!pip install torch transformers onnx

In [ ]:
from pathlib import Path

import torch
from transformers import AutoModelForQuestionAnswering, AutoTokenizer

MODEL_NAME = "distilbert-base-cased-distilled-squad"
MAX_LENGTH = 384

repo = Path.cwd()
onnx_path = repo / "distilbert_squad.onnx"
engine_path = repo / "distilbert_squad_trt" / "1" / "model.plan"
tokenizer_path = repo / "preprocess" / "1" / "tokenizer"
engine_path.parent.mkdir(parents=True, exist_ok=True)
tokenizer_path.mkdir(parents=True, exist_ok=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.save_pretrained(tokenizer_path)

model = AutoModelForQuestionAnswering.from_pretrained(MODEL_NAME).eval()

question = "What does Triton Control deploy?"
context = "Triton Control deploys NVIDIA Triton Inference Server instances."
encoded = tokenizer(
    question,
    context,
    max_length=MAX_LENGTH,
    padding="max_length",
    truncation="only_second",
    return_tensors="pt",
)

input_ids = encoded["input_ids"].to(torch.int32)
attention_mask = encoded["attention_mask"].to(torch.int32)

with torch.no_grad():
    torch.onnx.export(
        model,
        (input_ids, attention_mask),
        onnx_path,
        input_names=["input_ids", "attention_mask"],
        output_names=["start_logits", "end_logits"],
        opset_version=17,
        do_constant_folding=True,
    )

print(f"ONNX: {onnx_path}")
print(f"Tokenizer: {tokenizer_path}")
print(f"TensorRT plan target: {engine_path}")

In [ ]:
!trtexec \
  --onnx=distilbert_squad.onnx \
  --saveEngine=distilbert_squad_trt/1/model.plan \
  --fp16 \
  --shapes=input_ids:1x384,attention_mask:1x384

In [ ]:
from pathlib import Path

plan = Path("distilbert_squad_trt/1/model.plan")
tokenizer_config = Path("preprocess/1/tokenizer/tokenizer_config.json")
assert plan.exists(), "TensorRT plan was not created"
assert tokenizer_config.exists(), "Tokenizer files were not saved"
print(f"Created {plan} ({plan.stat().st_size / (1024 * 1024):.1f} MiB)")